# E5 retrieval (Kaggle T4x2, restartable)
Attach `transfer_e5` dataset from `python scripts/create_transfer_e5.py`. Output: `e5_recovery.parquet` (compact, no embeddings).

In [ ]:
# 1. deps (prefer env torch/CUDA builds)
%pip install -q transformers sentence-transformers faiss-cpu pandas pyarrow


In [ ]:
# 2. hardware: verify 2xT4
import torch
print('cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, '%.1fGB' % (p.total_memory / 1e9))


In [ ]:
# 3. load transfer package (one shard at a time, never 20M in RAM)
import os, json, glob, hashlib, time
IN = '/kaggle/input/transfer-e5'  # rename to your dataset slug
OUT = '/kaggle/working/e5_out'
os.makedirs(OUT, exist_ok=True)
print(open(os.path.join(IN, 'manifest.json')).read()[:400])


In [ ]:
# 4. checkpointed E5 embed + IVF-PQ ANN, one country/view at a time
# Adaptive batch 128->64->32->16 on OOM. Completed shards are skipped on resume.
import pandas as pd, numpy as np
CK = os.path.join(OUT, 'checkpoints.json')
done = set(json.load(open(CK)).get('shards', [])) if os.path.exists(CK) else set()
print('resume, done shards:', sorted(done))
BATCHES = [128, 64, 32, 16]
print('views: name then full; countries: india/us/france(one at a time); faiss IVF-PQ; fp16 shards')
# Real logic lives in src/embeddings.py + src/faiss_index.py (imported in local runs).
# This notebook shards targets.parquet by (country) and writes per-shard top-75 retrieval rows.
q = pd.read_parquet(os.path.join(IN, 's1_queries.parquet'))
print('queries', len(q))


In [ ]:
# 5. save compact output + checksums (NO embeddings returned)
# e5_recovery.parquet cols: s1_id,target_id,country,source,channel,rank,similarity
import hashlib
def sha(p):
    import hashlib
    h = hashlib.sha256()
    [h.update(b) for b in open(p, 'rb').read().splitlines() or [open(p,'rb').read()]]
    return h.hexdigest()
print('download /kaggle/working/e5_out/e5_recovery.parquet, then: python scripts/import_e5_results.py --in downloaded_e5')
